# 01 - Chunking and Embedding

**Goal:** turn the parsed 10-K elements (`data/parsed/*.elements.jsonl`) into
*chunks* that can be embedded and searched.

## The big picture

```
elements.jsonl --> chunks (text + metadata) --> embeddings (vectors) --> vector search
  (parsed)          [this notebook, part 1]      [part 2]                 [part 3]
```

Typical sweet spot: **~200-500 tokens** (roughly 800-2000 characters).

### Our strategy: structure-aware chunking
Because the parser already gave us structure (Part / Item / headings / tables), we do NOT blindly
slice every N characters. Instead:
1. **Never cross an Item boundary** (Risk Factors never mixes with MD&A).
2. **Pack paragraphs** under the same heading until we hit a size budget.
3. **Tables are atomic** - a table is its own chunk (split by rows only if huge, repeating the header).
4. **Add a breadcrumb** (`AAPL 10-K 2025 > Item 1A > Risk Factors > ...`) so each chunk is self-contained.

We start with the naive approach first, so you can see *why* the better one is needed.

In [1]:
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected; keep using the fastembed CPU path above.")

# GPU_MODEL_NAME = "BAAI/bge-small-en-v1.5"
GPU_MODEL_NAME = "BAAI/bge-base-en-v1.5"
gpu_device = "cuda"
print(torch.cuda.get_device_name(0))
gpu_model = SentenceTransformer(GPU_MODEL_NAME, device=gpu_device)

c:\Users\Satyajit_Khot\Downloads\techfiling-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 660.82it/s]


In [5]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from datetime import datetime

 
ROOT = Path.cwd()
if ROOT.name == "exploration":
    ROOT = ROOT.parent
 
PARSED = ROOT / "data" / "parsed"
 
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT = ROOT / "data" / "chunks" / RUN_TIMESTAMP
OUT.mkdir(parents=True, exist_ok=False)

def load_elements(doc_id):
    return [json.loads(l) for l in (PARSED / f"{doc_id}.elements.jsonl").open(encoding="utf-8")]

DOC_IDS = sorted(p.name.removesuffix(".elements.jsonl") for p in PARSED.glob("*.elements.jsonl"))
print(DOC_IDS)

['AAPL_10-K_2025', 'MSFT_10-K_2026', 'TSLA_10-K_2025']


# Step 1 : Look at input.

In [7]:
els_AAPL = load_elements("AAPL_10-K_2025")
els_MSFT = load_elements("MSFT_10-K_2026")
els_TSLA = load_elements("TSLA_10-K_2025")

all_docs = {
    "AAPL_10-K_2025": els_AAPL,
    "MSFT_10-K_2026": els_MSFT,
    "TSLA_10-K_2025": els_TSLA,
}

def check_unique(els):
    df = pd.DataFrame(els)
    # print(f"checking unique types : {df.type.value_counts().to_dict()}")
    # print(f"checking unique part : {df.part.value_counts().to_dict()}")
    print(f"checking unique part : {df.section_title.value_counts().to_dict()}")


for i in all_docs.values():
    check_unique(i)

checking unique part : {'Financial Statements and Supplementary Data': 214, 'Risk Factors': 107, 'Management’s Discussion and Analysis of Financial Condition and Results of Operations': 107, 'Business': 53, 'Cover Page': 43, 'Exhibit and Financial Statement Schedules': 15, 'Controls and Procedures': 13, 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities': 11, 'Legal Proceedings': 9, 'Quantitative and Qualitative Disclosures About Market Risk': 9, 'Form 10-K Summary': 9, 'Cybersecurity': 5, 'Other Information': 4, 'Unresolved Staff Comments': 2, 'Properties': 2, 'Mine Safety Disclosures': 2, 'Changes in and Disagreements with Accountants on Accounting and Financial Disclosure': 2, 'Disclosure Regarding Foreign Jurisdictions that Prevent Inspections': 2, 'Directors, Executive Officers and Corporate Governance': 2, 'Executive Compensation': 2, 'Security Ownership of Certain Beneficial Owners and Management and Related Stockholder 

In [8]:

def check_nulls(els, doc_id):
    df = pd.DataFrame(els)

    print(f"\n{'=' * 70}")
    print(f"Document: {doc_id}")
    print(f"Elements: {len(df)}")
    print(f"{'=' * 70}")

    null_report = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_percent": (df.isna().mean() * 100).round(2),
        "non_null_count": df.notna().sum(),
    })

    print(null_report)

    print("\nColumns containing null values:")
    print(null_report[null_report["null_count"] > 0])

    print("\nRows with null metadata:")
    metadata_keys = [
        "doc_id",
        "order",
        "type",
        "part",
        "section",
        "section_title",
    ]
    existing_keys = [key for key in metadata_keys if key in df.columns]

    null_rows = df[df[existing_keys].isna().any(axis=1)]
    display(null_rows[existing_keys])


for doc_id, elements in all_docs.items():
    check_nulls(elements, doc_id)


Document: AAPL_10-K_2025
Elements: 624
               null_count  null_percent  non_null_count
doc_id                  0          0.00             624
order                   0          0.00             624
type                    0          0.00             624
part                    0          0.00             624
section                 4          0.64             620
section_title           0          0.00             624
text                   51          8.17             573
table                 573         91.83              51

Columns containing null values:
         null_count  null_percent  non_null_count
section           4          0.64             620
text             51          8.17             573
table           573         91.83              51

Rows with null metadata:


,doc_id,order,type,part,section,section_title
43,AAPL_10-K_2025,43,heading,Part I,NaN,Part I
224,AAPL_10-K_2025,224,heading,Part II,NaN,Part II
588,AAPL_10-K_2025,588,heading,Part III,NaN,Part III
599,AAPL_10-K_2025,599,heading,Part IV,NaN,Part IV



Document: MSFT_10-K_2026
Elements: 967
               null_count  null_percent  non_null_count
doc_id                  0          0.00             967
order                   0          0.00             967
type                    0          0.00             967
part                    0          0.00             967
section                 4          0.41             963
section_title           0          0.00             967
text                   85          8.79             882
table                 882         91.21              85

Columns containing null values:
         null_count  null_percent  non_null_count
section           4          0.41             963
text             85          8.79             882
table           882         91.21              85

Rows with null metadata:


,doc_id,order,type,part,section,section_title
34,MSFT_10-K_2026,34,heading,Part I,NaN,Part I
325,MSFT_10-K_2026,325,heading,Part II,NaN,Part II
930,MSFT_10-K_2026,930,heading,Part III,NaN,Part III
945,MSFT_10-K_2026,945,heading,Part IV,NaN,Part IV



Document: TSLA_10-K_2025
Elements: 957
               null_count  null_percent  non_null_count
doc_id                  0          0.00             957
order                   0          0.00             957
type                    0          0.00             957
part                    0          0.00             957
section                 4          0.42             953
section_title           0          0.00             957
text                   74          7.73             883
table                 883         92.27              74

Columns containing null values:
         null_count  null_percent  non_null_count
section           4          0.42             953
text             74          7.73             883
table           883         92.27              74

Rows with null metadata:


,doc_id,order,type,part,section,section_title
38,TSLA_10-K_2025,38,heading,Part I,NaN,Part I
315,TSLA_10-K_2025,315,heading,Part II,NaN,Part II
924,TSLA_10-K_2025,924,heading,Part III,NaN,Part III
935,TSLA_10-K_2025,935,heading,Part IV,NaN,Part IV


In [9]:

def check_distribution(els, doc_id):
    df = pd.DataFrame(els)

    print(f"\n{'=' * 70}")
    print(f"Document: {doc_id}")
    print(f"Elements: {len(df)}")
    print(f"{'=' * 70}")
    print("Printing unique types")
    print(df.type.value_counts().to_dict())
    print(f"Printing distribution of paragraph ")
    print(df.assign(chars=df.text.fillna("").str.len()).query("type=='paragraph'").chars.describe() )
    print("*"*80)
    print(f"Printing distribution of headings ")
    print(df.assign(chars=df.text.fillna("").str.len()).query("type=='heading'").chars.describe() )
    print("*"*80)
    table_els = [e for e in els if e['type'] == 'table']
    print(f"Printing distribution of table ")
    markdown_lengths = [len(e['table']['markdown']) for e in table_els]
    
    # Show statistics
    print(pd.Series(markdown_lengths).describe())


for doc_id, elements in all_docs.items():
    check_distribution(elements, doc_id)


Document: AAPL_10-K_2025
Elements: 624
Printing unique types
{'paragraph': 425, 'heading': 148, 'table': 51}
Printing distribution of paragraph 
count     425.000000
mean      382.200000
std       423.840527
min         2.000000
25%        87.000000
50%       239.000000
75%       560.000000
max      3647.000000
Name: chars, dtype: float64
********************************************************************************
Printing distribution of headings 
count    148.000000
mean      36.256757
std       28.343165
min        6.000000
25%       17.750000
50%       27.000000
75%       47.000000
max      149.000000
Name: chars, dtype: float64
********************************************************************************
Printing distribution of table 
count      51.000000
mean      905.607843
std       970.069504
min       192.000000
25%       336.500000
50%       511.000000
75%      1004.000000
max      4450.000000
dtype: float64

Document: MSFT_10-K_2026
Elements: 967
Printing unique ty

# Step 2 : Structure-aware chunker

Algorithm (single pass over the elements, in order):
- Keep a **buffer** of paragraph text and the **current heading trail**.
- `heading`  -> if the buffer already has body text, flush it (new topic); then remember the heading.
- `paragraph`-> add to buffer; if buffer would exceed `MAX_CHARS`, flush first.
- `table`    -> flush buffer, emit table chunk(s).
- Section (Item) changes -> flush.
- A single paragraph bigger than `MAX_CHARS` is split on sentence boundaries.

Each chunk stores:
- `text`: what we show to the LLM,
- `embed_text`: breadcrumb + text, what we actually embed (gives the vector the context),
- metadata for filtering/citations (`doc_id`, `part`, `section`, `element_orders`, `type`).

In [10]:
els = load_elements("AAPL_10-K_2025")
# below sizes are considering breadcrumb and text 
MAX_CHARS = 1300
MIN_CHARS = 200
TABLE_MAX_CHARS = 1000


#Changing above sizes because of token ration is 3.23 and with above size we are crossing limit of 512 token 

SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z(\"'])")

def split_long(text, max_chars):
    """Split an oversized paragraph on sentence boundaries."""
    parts, cur = [], ""
    for s in SENT_SPLIT.split(text):
        if cur and len(cur) + len(s) + 1 > max_chars:
            parts.append(cur)
            cur = s
        else:
            cur = f"{cur} {s}".strip()
    if cur:
        parts.append(cur)
    return parts

def concise_table_intro(intro, max_chars=300):
    intro = (intro or "").strip()
    if not intro:
        return ""
    sentences = [s.strip() for s in SENT_SPLIT.split(intro) if s.strip()]
    if not sentences:
        return intro[-max_chars:]
    selected, total = [], 0
    for sentence in reversed(sentences):
        additional = len(sentence) + (1 if selected else 0)
        if selected and total + additional > max_chars:
            break
        selected.append(sentence)
        total += additional
    return " ".join(reversed(selected))


def split_table_markdown(md, max_chars):
    """Split a markdown table by rows, repeating title + header on every piece."""
    if len(md) <= max_chars:
        return [md]
    lines = md.split("\n")
    sep = next(i for i, l in enumerate(lines) if re.fullmatch(r"\|[-| ]+\|", l))
    head, rows = lines[:sep + 1], lines[sep + 1:]
    pieces, cur = [], []
    for r in rows:
        if cur and len("\n".join(head + cur + [r])) > max_chars:
            pieces.append("\n".join(head + cur))
            cur = []
        cur.append(r)
    if cur:
        pieces.append("\n".join(head + cur))
    return pieces

In [11]:
def breadcrumb(e, headings):
    parts = [e["doc_id"].replace("_", " "), e.get("part"), e.get("section"), e.get("section_title")]
    trail = [p for p in parts if p] + headings[-2:]
    return " > ".join(trail)

def chunk_document(els, max_chars=MAX_CHARS):
    chunks, buf, buf_orders, headings, pending = [], [], [], [], []
    state = {"key": None, "first": None}

    def emit(type_, text, orders, ref, extra=None):
        c = {
            "chunk_id": f"{ref['doc_id']}_c{len(chunks):04d}",
            "doc_id": ref["doc_id"], "part": ref.get("part"), "section": ref.get("section"),
            "section_title": ref.get("section_title"), "type": type_,
            "headings": list(headings[-2:]), "element_orders": orders,
            "text": text, "breadcrumb": breadcrumb(ref, headings),
        }
        c["embed_text"] = f"{c['breadcrumb']}\n\n{text}"
        c.update(extra or {})
        chunks.append(c)

    def flush(ref):
        if buf:
            emit("text", "\n\n".join(buf), buf_orders[:], ref)
        buf.clear(); buf_orders.clear()

    for e in els:
        key = (e["part"], e["section"])
        if key != state["key"]:                      # new Item -> hard boundary
            if state["first"]: flush(state["first"])
            headings.clear(); state["key"] = key
        state["first"] = e
        ref = e

        if e["type"] == "heading":
            if buf: flush(ref)                        # heading after body text = new topic
            headings.append(e["text"])
            continue

        if e["type"] == "table":
            flush(ref)
            t = e["table"]
            full_intro = t.get("intro")
            table_context = concise_table_intro(full_intro)
            for piece in split_table_markdown(t["markdown"], TABLE_MAX_CHARS):
                text = f"{table_context}\n\n{piece}" if table_context else piece
                emit("table", text, [e["order"]], ref,
                     {"table_id": t["table_id"], "table_title": t.get("title"), "units": t.get("units"),
                      "intro": full_intro, "table_context": table_context})
            continue

        text = (e["text"] or "").strip()
        if not text:
            continue
        for piece in (split_long(text, max_chars) if len(text) > max_chars else [text]):
            if buf and sum(map(len, buf)) + len(piece) > max_chars:
                flush(ref)
            buf.append(piece); buf_orders.append(e["order"])
    if state["first"]: flush(state["first"])
    return chunks

chunks = chunk_document(els)
print(len(chunks), "chunks")
pd.Series([c["type"] for c in chunks]).value_counts().to_dict()

324 chunks


{'text': 246, 'table': 78}

In [12]:
print(chunks[90]['embed_text'])

AAPL 10-K 2025 > Part I > Item 1A > Risk Factors > The Company’s business and reputation are impacted by information technology system failures and network disruptions. > Legal and Regulatory Compliance Risks

The Company is also subject to new and changing laws and regulations regarding online safety, including enhanced protections for minors and mandatory age verification requirements. These laws and regulations can increase regulatory risks by requiring complex compliance measures and significant modifications to the Company’s products, services and operations, and may lead to operational disruptions, heightened privacy and data security risks, increased costs and potential liability and fines, all of which can have a material adverse impact on the Company’s business, financial condition, results of operations and stock price.


In [13]:
# Need to check size of embed_text
embed_chunks = [ len(i['embed_text']) for i in chunks ]
# markdown_lengths = [len(e['table']['markdown']) for e in table_els]
pd.Series(embed_chunks).describe()

count     324.000000
mean      841.070988
std       397.051670
min        90.000000
25%       495.000000
50%       893.500000
75%      1160.500000
max      1584.000000
dtype: float64

In [14]:
# Inspect a few chunks: one text chunk, one table chunk
txt = next(c for c in chunks if c["type"] == "text" and c["section"] == "Item 1A" and len(c["text"]) > 800)
tab = next(c for c in chunks if c["type"] == "table")
for c in (txt, tab):
    print("=" * 90)
    print("EMBED_TEXT:\n", c["embed_text"][:900])
    print({k: c[k] for k in ("chunk_id", "section", "headings", "element_orders")})

EMBED_TEXT:
 AAPL 10-K 2025 > Part I > Item 1A > Risk Factors > Item 1A. Risk Factors

The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement of all potential risks or uncertainties that the Company faces or may face in the future.

This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financi
{'chunk_id': 'AAPL_10-K_2025_c0031', 'section': 'Item 1A', 'headings': ['Item 1A. Risk

## Step 3 - Evaluate the chunks
Always look at the size distribution. Too many tiny chunks (cover page lines, one-line headings)
are noise for retrieval.

In [15]:
def stats(chunks):
    s = pd.DataFrame({"type": [c["type"] for c in chunks], "chars": [len(c["embed_text"]) for c in chunks]})
    return s.groupby("type").chars.describe().round(0)

stats(chunks)

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
table,78.0,904.0,263.0,384.0,678.0,954.0,1117.0,1329.0
text,246.0,821.0,429.0,90.0,365.0,853.0,1189.0,1584.0


In [16]:
tiny = [c for c in chunks if len(c["text"]) < MIN_CHARS]
print(len(tiny), "tiny chunks of", len(chunks))
for c in tiny[:8]:
    print(c["section"], "|", repr(c["text"][:90]))

59 tiny chunks of 324
Cover Page | '(Registrant’s telephone number, including area code)\n\nSecurities registered pursuant to Se'
Item 1 | 'Payment Services\n\nThe Company offers payment services, including Apple Card®, a co-branded'
Item 1A | 'If the Company is unable to compete successfully, its business, reputation, results of ope'
Item 1A | 'The Company depends on component and product manufacturing and logistical services provide'
Item 1A | 'The Company’s gross margins vary significantly across its products, services, geographic s'
Item 1B | 'None.'
Item 4 | 'Not applicable.'
Item 5 | 'The Company’s common stock is traded on The Nasdaq Stock Market LLC under the symbol AAPL.'


In [17]:
def inspect_tiny_chunks(chunks, min_chars):
    tiny = [
        c for c in chunks
        if len(c.get("text", "").strip()) < min_chars
    ]

    print(f"{len(tiny)} tiny chunks out of {len(chunks)}")
    print(f"{len(tiny) / max(len(chunks), 1):.1%} of all chunks")

    for c in tiny[:20]:
        text = c.get("text", "").strip()

        print(
            f"\ndoc_id: {c.get('doc_id')}"
            f"\npart: {c.get('part')}"
            f"\nsection: {c.get('section')}"
            f"\nsection_title: {c.get('section_title')}"
            f"\ntype: {c.get('chunk_type', c.get('type'))}"
            f"\ncharacters: {len(text)}"
            f"\ntext: {text!r}"
        )

    return tiny


tiny = inspect_tiny_chunks(chunks, MIN_CHARS)

59 tiny chunks out of 324
18.2% of all chunks

doc_id: AAPL_10-K_2025
part: Front Matter
section: Cover Page
section_title: Cover Page
type: text
characters: 113
text: '(Registrant’s telephone number, including area code)\n\nSecurities registered pursuant to Section 12(b) of the Act:'

doc_id: AAPL_10-K_2025
part: Part I
section: Item 1
section_title: Business
type: text
characters: 147
text: 'Payment Services\n\nThe Company offers payment services, including Apple Card®, a co-branded credit card, and Apple Pay®, a cashless payment service.'

doc_id: AAPL_10-K_2025
part: Part I
section: Item 1A
section_title: Risk Factors
type: text
characters: 172
text: 'If the Company is unable to compete successfully, its business, reputation, results of operations, financial condition and stock price can be materially adversely affected.'

doc_id: AAPL_10-K_2025
part: Part I
section: Item 1A
section_title: Risk Factors
type: text
characters: 162
text: 'The Company depends on component and product m

In [18]:

def merge_tiny(chunks, min_chars=MIN_CHARS, max_chars=MAX_CHARS):
    out = []
    for c in chunks:
        p = out[-1] if out else None
        if (p and p["type"] == "text" and c["type"] == "text" and c["section"] == p["section"]
                and c["doc_id"] == p["doc_id"]
                and (len(p["text"]) < min_chars) and len(p["text"]) + len(c["text"]) <= max_chars):
            p["text"] += "\n\n" + c["text"]
            p["element_orders"] += c["element_orders"]
            p["embed_text"] = f"{p['breadcrumb']}\n\n{p['text']}"
        else:
            out.append(c)
    for i, c in enumerate(out):                      # renumber ids
        c["chunk_id"] = f"{c['doc_id']}_c{i:04d}"
    return out

merged = merge_tiny(chunks)
print(len(chunks), "->", len(merged))
stats(merged)

324 -> 304


,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
table,78.0,904.0,263.0,384.0,678.0,954.0,1117.0,1329.0
text,226.0,880.0,406.0,105.0,526.0,926.0,1214.0,1584.0


# In future we may need to remove all tiny chunks

# Step 4  Chunk all filings and save
Output: `data/chunks/<run_timestamp>/<doc_id>.chunks.jsonl` plus `run_manifest.json`.

In [19]:
all_chunks = []
for d in DOC_IDS:
    cs = merge_tiny(chunk_document(load_elements(d)))
    with (OUT / f"{d}.chunks.jsonl").open("w", encoding="utf-8") as f:
        for c in cs:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    all_chunks += cs
    print(d, len(cs))
print("total", len(all_chunks))

AAPL_10-K_2025 304
MSFT_10-K_2026 472
TSLA_10-K_2025 531
total 1307


# Step 5 : Embeddings   [ Using GPU encoding with Sentence Transformers] 


#### Before embedding let's verify does our chunks get tructed after embedding 


In [20]:
embed_chunk_tokens = [len(gpu_model.tokenizer.encode(c['embed_text'], add_special_tokens=False)) for c in chunks]
pd.Series(embed_chunk_tokens).describe()

count    324.000000
mean     188.598765
std       88.305897
min       29.000000
25%      116.000000
50%      191.500000
75%      257.000000
max      437.000000
dtype: float64

In [21]:
print("Embedding dimensions:", gpu_model.get_sentence_embedding_dimension())
print("Model max sequence length:", gpu_model.max_seq_length)
print("Tokenizer max length:", gpu_model.tokenizer.model_max_length)

Embedding dimensions: 768
Model max sequence length: 512
Tokenizer max length: 512


C:\Users\Satyajit_Khot\AppData\Local\Temp\ipykernel_35268\864703687.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", gpu_model.get_sentence_embedding_dimension())


1. model max sequence length is 512 but our some chunk has max token 785 so some information may get trunted 

In [22]:
#Checking which chunk is getting overhead 
MAX_TOKENS = gpu_model.max_seq_length

for c in chunks:
    c["embed_tokens"] = len(
        gpu_model.tokenizer.encode(
            c["embed_text"],
            add_special_tokens=True,
            truncation=False,
        )
    )

oversized = [c for c in chunks if c["embed_tokens"] > MAX_TOKENS]

print(f"{len(oversized)} oversized chunks out of {len(chunks)}")

for c in sorted(
    oversized,
    key=lambda x: x["embed_tokens"],
    reverse=True,
)[:20]:
    print(
        c["embed_tokens"],
        "|",
        c.get("doc_id"),
        "|",
        c.get("section"),
        "|",
        c.get("chunk_type"),
        "|",
        c.get("element_orders")
    )

0 oversized chunks out of 324


In [23]:
print(merged[0].keys())

dict_keys(['chunk_id', 'doc_id', 'part', 'section', 'section_title', 'type', 'headings', 'element_orders', 'text', 'breadcrumb', 'embed_text', 'embed_tokens'])


In [24]:
for chunk in merged:
    if 608 in chunk.get("element_orders", []):
        print(chunk["text"])


**Table**

| Exhibit Number | Exhibit Description | Incorporated by Reference Form | Incorporated by Reference Exhibit | Incorporated by Reference Filing Date/ Period End Date |
|---|---|---|---|---|
| 4.9 | Officer’s Certificate of the Registrant, dated as of September 17, 2015, including forms of global notes representing the 1.375% Notes due 2024 and 2.000% Notes due 2027. | 8-K | 4.1 | 9/17/15 |
| 4.10 | Officer’s Certificate of the Registrant, dated as of February 23, 2016, including forms of global notes representing the Floating Rate Notes due 2019, Floating Rate Notes due 2021, 1.300% Notes due 2018, 1.700% Notes due 2019, 2.250% Notes due 2021, 2.850% Notes due 2023, 3.250% Notes due 2026, 4.500% Notes due 2036 and 4.650% Notes due 2046. | 8-K | 4.1 | 2/23/16 |
| 4.11 | Supplement No. 1 to the Officer’s Certificate of the Registrant, dated as of March 24, 2016. | 8-K | 4.1 | 3/24/16 |
**Table**

| Exhibit Number | Exhibit Description | Incorporated by Reference Form | Incorpor

In [25]:
MODEL_MAX_TOKENS = int(min(
    gpu_model.max_seq_length,
    gpu_model.tokenizer.model_max_length,
))
EMBED_TOKEN_LIMIT = int(MODEL_MAX_TOKENS * 0.94)
EMBEDDING_DIMENSION = int(gpu_model.get_embedding_dimension())

for chunk in all_chunks:
    chunk["embed_tokens"] = len(gpu_model.tokenizer.encode(
        chunk["embed_text"],
        add_special_tokens=True,
        truncation=False,
    ))

def token_summary(values):
    series = pd.Series(values)
    return {
        "count": int(series.count()),
        "mean": round(float(series.mean()), 2),
        "std": round(float(series.std()), 2),
        "min": int(series.min()),
        "p25": float(series.quantile(0.25)),
        "p50": float(series.quantile(0.50)),
        "p75": float(series.quantile(0.75)),
        "max": int(series.max()),
    }

def chunk_summary(doc_chunks):
    token_counts = [c["embed_tokens"] for c in doc_chunks]
    type_counts = pd.Series([c["type"] for c in doc_chunks]).value_counts().to_dict()
    return {
        "chunks": len(doc_chunks),
        "chunks_by_type": {key: int(value) for key, value in type_counts.items()},
        "tiny_chunks": sum(len(c["text"].strip()) < MIN_CHARS for c in doc_chunks),
        "over_recommended_limit": sum(n > EMBED_TOKEN_LIMIT for n in token_counts),
        "over_model_limit": sum(n > MODEL_MAX_TOKENS for n in token_counts),
        "token_statistics": token_summary(token_counts),
    }

manifest = {
    "run_timestamp": RUN_TIMESTAMP,
    "output_directory": str(OUT.relative_to(ROOT)),
    "chunking_config": {
        "max_chars": MAX_CHARS,
        "min_chars": MIN_CHARS,
        "table_max_chars": TABLE_MAX_CHARS,
    },
    "embedding_model": {
        "name": GPU_MODEL_NAME,
        "dimension": EMBEDDING_DIMENSION,
        "model_max_tokens": int(gpu_model.max_seq_length),
        "tokenizer_max_tokens": int(gpu_model.tokenizer.model_max_length),
        "effective_max_tokens": MODEL_MAX_TOKENS,
        "recommended_token_limit": EMBED_TOKEN_LIMIT,
    },
    "overall": chunk_summary(all_chunks),
    "documents": {
        doc_id: chunk_summary([c for c in all_chunks if c["doc_id"] == doc_id])
        for doc_id in DOC_IDS
    },
}

manifest_path = OUT / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"Manifest written to {manifest_path}")

{
  "run_timestamp": "2026-10-16_14-16-07",
  "output_directory": "data\\chunks\\2026-10-16_14-16-07",
  "chunking_config": {
    "max_chars": 1300,
    "min_chars": 200,
    "table_max_chars": 1000
  },
  "embedding_model": {
    "name": "BAAI/bge-base-en-v1.5",
    "dimension": 768,
    "model_max_tokens": 512,
    "tokenizer_max_tokens": 512,
    "effective_max_tokens": 512,
    "recommended_token_limit": 481
  },
  "overall": {
    "chunks": 1307,
    "chunks_by_type": {
      "text": 999,
      "table": 308
    },
    "tiny_chunks": 107,
    "over_recommended_limit": 0,
    "over_model_limit": 0,
    "token_statistics": {
      "count": 1307,
      "mean": 196.12,
      "std": 79.82,
      "min": 31,
      "p25": 138.0,
      "p50": 201.0,
      "p75": 252.5,
      "max": 439
    }
  },
  "documents": {
    "AAPL_10-K_2025": {
      "chunks": 304,
      "chunks_by_type": {
        "text": 226,
        "table": 78
      },
      "tiny_chunks": 32,
      "over_recommended_limit": 0,

In [26]:
tesla_oversized = [
    c for c in all_chunks
    if c["doc_id"] == "TSLA_10-K_2025"
    and c["embed_tokens"] > MODEL_MAX_TOKENS
]

for c in sorted(
    tesla_oversized,
    key=lambda x: x["embed_tokens"],
    reverse=True,
):
    print("\n" + "=" * 100)
    print("chunk_id:", c["chunk_id"])
    print("type:", c["type"])
    print("tokens:", c["embed_tokens"])
    print("element_orders:", c["element_orders"])
    print("table_id:", c.get("table_id"))
    print(c["text"][:1000])

In [27]:
# let's do embedding 
gpu_embeddings = gpu_model.encode(
    [c["embed_text"] for c in all_chunks],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")
print(gpu_embeddings.shape)
np.save(OUT / "embeddings_gpu_base.npy", gpu_embeddings)

Batches: 100%|██████████| 21/21 [00:18<00:00,  1.12it/s]

(1307, 768)


In [28]:
def show(hits, width=350):
    for s, c in hits:
        print(f"{s:.3f} | {c['breadcrumb']} [{c['type']}]")
        print("      ", c["text"][:width].replace("\n", " "))

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def search_gpu(query, k=5, doc_id=None, section=None, type_=None):
    q = gpu_model.encode(
        BGE_QUERY_PREFIX + query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores = gpu_embeddings @ q
    order = np.argsort(-scores)
    hits = []
    for i in order:
        c = all_chunks[i]
        if doc_id and c["doc_id"] != doc_id: continue
        if section and c["section"] != section: continue
        if type_ and c["type"] != type_: continue
        hits.append((float(scores[i]), c))
        if len(hits) == k: break
    return hits

show(search_gpu("What was Apple's total net sales in 2025?", doc_id="AAPL_10-K_2025"),900)

0.764 | AAPL 10-K 2025 > Part II > Item 7 > Management’s Discussion and Analysis of Financial Condition and Results of Operations > Segment Operating Performance > Products and Services Performance [text]
       (1)Services net sales include amortization of the deferred value of services bundled in the sales price of certain products.  iPhone  iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models.  Mac  Mac net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops.  iPad  iPad net sales increased during 2025 compared to 2024 primarily due to higher net sales of iPad Air, iPad mini and iPad, partially offset by lower net sales of iPad Pro.  Wearables, Home and Accessories  Wearables, Home and Accessories net sales decreased during 2025 compared to 2024 primarily due to lower net sales of Accessories and Wearables.  Services  Services net sales increased during 2025 compared to 2024 primarily due to 